<a id="local-execution"></a>
## Hands-On: Running a Quantized Model Locally

Now that we know how to read a Hugging Face Model Card, let's actually download a model and run it entirely on our own hardware. No API keys, no internet connection required (after the initial download), and complete privacy for your data.

To do this, we use the `transformers` library, which is the industry standard Python package built by Hugging Face.

### The Pipeline: Tokenizer + Model



When running a model locally, you must load two separate components:
1. **The Tokenizer:** Models don't read English; they read numbers. The tokenizer translates your text string into a mathematical sequence (tokens).
2. **The Model:** The actual neural network (the parameters/weights) that processes the numbers and predicts the next numbers.

### The Plan: 4-Bit Quantization on a 0.5B Model
For this exercise, we will use `Qwen/Qwen2.5-0.5B-Instruct`. This is an open, "ultra-small" instruction-tuned model. 

Even though 0.5B is tiny, we are going to apply **4-bit quantization** using a library called `bitsandbytes`. This will shrink the model's memory footprint so aggressively that it can run on almost any standard student laptop without crashing the system.

---
💡 **Tip:** If you were running a massive 70B model, the code below is *exactly the same*. You would just change the model name string!

In [1]:
# Install the required Hugging Face libraries
# - transformers: To load the model and tokenizer
# - accelerate: Helps manage computer memory efficiently
# - bitsandbytes: The engine that performs the 4-bit quantization
!pip install -q transformers accelerate bitsandbytes


[notice] A new release of pip is available: 25.0.1 -> 26.0.1
[notice] To update, run: pip install --upgrade pip


### Step 1: Configure the "Shrink Ray" (Quantization)

Before we download the model, we have to tell Python *how* to load it. If we don't specify quantization, it will load in full 32-bit precision and eat up all your RAM. 

We will use `BitsAndBytesConfig` to force the model to load in 4-bit precision.

In [2]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

# 1. Define the model we want from Hugging Face
model_id = "Qwen/Qwen2.5-0.5B-Instruct"

# 2. Configure 4-bit Quantization
quantization_config = BitsAndBytesConfig(
    load_in_4bit=True, # Turn on 4-bit compression
    bnb_4bit_compute_dtype=torch.float16 # Keep computations relatively fast
)

print(f"Downloading tokenizer for {model_id}...")
# 3. Download and load the Tokenizer
tokenizer = AutoTokenizer.from_pretrained(model_id)

print(f"Downloading and quantizing model weights for {model_id} (This may take a minute)...")
# 4. Download and load the Model (applying our quantization config)
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=quantization_config,
    device_map="auto" # Automatically put the model on GPU if available, or CPU if not
)

print("Model loaded successfully in 4-bit precision!")

config.json:   0%|          | 0.00/659 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/988M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Model loaded successfully in 4-bit precision!


In [4]:
inputs = tokenizer.apply_chat_template(
    messages,
    add_generation_prompt=True,
    tokenize=True,
    return_dict=True,
    return_tensors="pt",
).to(model.device)

outputs = model.generate(
    **inputs,
    max_new_tokens=200,
    temperature=0.7
)

response = tokenizer.decode(
    outputs[0][inputs["input_ids"].shape[-1]:],
    skip_special_tokens=True
)

print(response)


: 

: 

: 

### 🥊 Challenge 3 (10 minutes): Test the Limits of an Ultra-Small Model

You just ran a 0.5-Billion parameter model. Because it is highly quantized and very small, its reasoning abilities are limited. 

**Your Task:**
1. Modify the `messages` list in the Python cell above.
2. Give the model a complex logic puzzle or a highly nuanced social science extraction task (e.g., *"Read this paragraph and extract the implicit bias of the author."*)
3. Run the cell.

**Discussion:** Did it succeed? Did it hallucinate? Where does the "ultra-small" model break down compared to the massive models we tested on OpenRouter earlier?

<div class="alert alert-warning">
⚠️ <b>The Local Tradeoff</b><br>

Running models locally gives you infinite, free generation and perfect privacy. However, you are strictly bottlenecked by the hardware in front of you. Social scientists must constantly weigh the privacy of local, small models against the high-tier reasoning of massive, API-hosted models.
</div>

In [1]:
# Install the Hugging Face evaluate library and the rouge_score dependency
!pip install -q evaluate rouge_score


[notice] A new release of pip is available: 25.0.1 -> 26.0.1
[notice] To update, run: pip install --upgrade pip


<a id="evaluation"></a>
## 7. Evaluation: Proving Your Model Works

Reading a few model outputs and thinking, *"Yeah, that looks pretty good,"* is fine for casual use. But in social science research, if you are going to use an LLM to code 10,000 qualitative surveys, you need hard, reproducible numbers to prove the model is accurate. 

How do we actually measure the performance of a language model against a human "Ground Truth" (a Golden Dataset)? 

---

### 7.1 Traditional NLP Metrics: The Word-Matchers
Before LLMs, researchers used statistical metrics to evaluate tasks like translation and summarization. We can calculate these easily using the Hugging Face `evaluate` library.

* **BLEU:** Measures **Precision**: How many word sequences in the AI's output explicitly appear in the human reference text? 
* **ROUGE:** Measures **Recall**: How much of the human reference text was successfully captured by the AI's output?

⚠️ **The Pitfall of Word-Matchers**
These metrics do not understand meaning; they only understand letters. Let's look at what happens when an LLM gives a conceptually perfect answer, but uses synonyms.

In [6]:
import evaluate

# 1. Load the metrics from Hugging Face
bleu_metric = evaluate.load("bleu")
rouge_metric = evaluate.load("rouge")

# 2. Define our texts. Notice how the LLM perfectly paraphrased the human!
human_reference = "The automobile is extremely quick."
llm_prediction = "The car is very fast."

# 3. Calculate BLEU
# Note: BLEU expects a list of lists for references (because a single prediction can have multiple valid references)
bleu_results = bleu_metric.compute(predictions=[llm_prediction], references=[[human_reference]])

# 4. Calculate ROUGE
rouge_results = rouge_metric.compute(predictions=[llm_prediction], references=[human_reference])

print("--- WORD-MATCHER RESULTS ---")
print(f"BLEU Score: {bleu_results['bleu']}")
print(f"ROUGE-L Score: {rouge_results['rougeL']}")
print("\nTakeaway: The model got a score of 0.0, even though it was conceptually correct! This is why word-matchers struggle with creative LLMs.")

--- WORD-MATCHER RESULTS ---
BLEU Score: 0.0
ROUGE-L Score: 0.4000000000000001

Takeaway: The model got a score of 0.0, even though it was conceptually correct! This is why word-matchers struggle with creative LLMs.


<a id="semantic-metrics"></a>
### 7.2 Deterministic Metrics (Structure)
If we can't rely on exact word matching for open-ended text, what *can* we use it for? 

For many social science tasks, we ask the LLM to extract structured data (like classifying sentiment as `Positive` or `Negative`). In these cases, we use a deterministic metric called **Exact Match**.

The `evaluate` library makes it easy to check for exact matches while safely ignoring minor formatting issues like capitalization or trailing spaces.

In [8]:
# Load the Exact Match metric
exact_match_metric = evaluate.load("exact_match")

# The ground truth we want the model to output
human_labels = ["Positive", "Negative", "Neutral"]

# The actual, slightly messy outputs generated by our LLM
llm_outputs = ["positive", "Negative", " neutral "]

# Compute the exact match, but tell Hugging Face to be forgiving with casing and spaces!
em_results = exact_match_metric.compute(
    predictions=[pred.strip() for pred in llm_outputs], #strip white space 
    references=human_labels,
    ignore_case=True, # Treats "Positive" and "positive" as identical
    ignore_punctuation=True
)

print(f"Exact Match Accuracy: {em_results['exact_match'] * 100}%")

Exact Match Accuracy: 100.0%


<a id="llm-judge"></a>
### 7.3 The Modern Standard: LLM-as-a-Judge



Because human evaluation is incredibly slow and metrics like BLEU are flawed for open text, the modern industry standard is **LLM-as-a-Judge**. 

Instead of writing code to evaluate nuance, researchers use a massive, highly capable API model (like `GPT-4o` or `Claude 3.5 Sonnet`) to grade the outputs of their smaller, locally running models.

**How it works:** You send the Judge LLM a prompt containing:
1. The original question.
2. A strict grading rubric (e.g., *"Score 1-5. A 5 must contain X and Y."*)
3. The human ground truth.
4. The small model's generated answer.

The Judge LLM then reads the small model's answer, critiques it against the rubric, and outputs a final numerical score (usually in JSON format so your Python script can easily record it!).

<a id="hf-leaderboard"></a>
## 8. Hugging Face & IFEval

While you can build custom metrics, you need a starting point to figure out which models are fundamentally capable. This is where the **Hugging Face Open LLM Leaderboard** comes in.



When researchers release a new open-source model, Hugging Face runs it through a gauntlet of standardized tests. For social scientists, the most critical benchmark on this leaderboard is **IFEval**.

### What is IFEval (Instruction Following Evaluation)?
IFEval is a test that completely ignores *what* the model says, and only grades *how* it formats the answer. 

It gives the model strict, verifiable constraints, such as:
* *"Write a 3-paragraph summary."*
* *"Do not use the letter 'e'."*
* *"Output your response as a valid JSON object."*

**Why it matters:** If a model has a low IFEval score, it will break your Python pipelines by refusing to output strict data formats! It might have brilliant reasoning, but if it wraps its answer in conversational fluff (*"Here is the JSON you requested!"*), your Exact Match evaluators will crash.

<a id="ifeval-challenge"></a>
### 🥊 Challenge 4 (10 minutes): The IFEval Test

Let's run a manual IFEval test right now to see how well models actually follow instructions.

**Your Task:**
1. Go to the official IFEval dataset on Hugging Face: [google/IFEval](https://huggingface.co/datasets/google/IFEval/viewer)
2. Look at the `prompt` column and find a highly restrictive prompt. (e.g., *"Write a 2 paragraph critique of the following sentence in all capital letters, no lowercase letters allowed..."*)
3. Copy that prompt.
4. Go to [Hugging Face Chat](https://huggingface.co/chat/) or your preferred LLM interface.
5. Paste the prompt into a large Instruct model (like Llama-3-70B) and generate a response.
6. **Evaluate:** Did it actually follow every single rule? Count the paragraphs. Check the capitalization. 

**Discussion:** Try running the exact same prompt on a much smaller model (like a 1B parameter model) or a Base model. Notice how quickly smaller models forget the formatting constraints halfway through their answer!

***
<div class="alert alert-success">  
🎉 <b>Congratulations!</b> You have completed the Decoding LLMs workshop. 

You can now navigate the AI ecosystem, understand the tradeoffs of model sizes and quantization, run models locally using `transformers`, and rigorously evaluate their outputs using the `evaluate` library and the Hugging Face Leaderboard. You are officially model literate!
</div>